# 实验一：模型评测流水线（Notebook 版）

> 对应课程：第1讲 · 对应项目：P1 · 四层归属：入门 · 预计耗时：5-6h（必做）+ 1-2h（选做）

**用法**：在带 `# SUBMIT` 标记的代码单元格里填代码（`# FIXED`、导出、评分单元格不要动），
每填完一关运行旁边的「自测」单元格验证；全部完成后**先保存 notebook（Ctrl+S / File > Save），
再运行最底部的「评分」单元格**——评分脚本直接读取本 notebook 的最新保存内容
（按 `# FIXED / # SUBMIT` 标记拼装代码、按 `<!-- REPORT -->` 单元格取报告），**不需要先导出**；
「导出」单元格用于契约自检与生成 `.py` 版，与 `.py` 版讲义使用**同一个评分脚本、同一套函数契约**。

遇到没见过的函数/对象，先看下面的「API 速查与自查方法」单元格；做完实验可以翻到末尾「课本思考题」消化学过的东西。

## 实验目标

1. 掌握 LangChain 的 Chat Model API（同步、流式、结构化输出）
2. 学会测量 LLM 的关键性能指标（TTFT、耗时、Token 用量）
3. 建立一个可复用的模型评测流程

## 前置准备

```bash
pip install langchain langchain-core python-dotenv
```

- 在本 notebook 同目录放一个 `.env`：`DEEPSEEK_API_KEY=sk-your-key`
- 把评分脚本 `lab01_test.py` 复制到本 notebook 同目录（评分按当前目录查找交付物）
- 模型固定为 `deepseek-chat`，**不得修改模型名和参数**（公平规则）

## 闯关规则

- 必做 L1-L5 累计 100 分；选做 L6/L7 各 10 分加分，总分上限 120（不做选做不扣必做分）；评分脚本 import 你的模块后**逐关调用约定函数**——契约以各单元格 docstring 为准
- L1 在函数内部自建模型；L2/L3/L4 由评分脚本传入 `model` 参数
- 可反复提交，取**历史最高分**
- 跨模型对比自练（不计分）：另申请 `glm-4-flash` Key——交付数据只收 `deepseek-chat`

| 关卡 | 分值 | 核心要求 |
|------|------|---------|
| L1 基础调用 | 10 | `basic_call` 调通模型，输出切题非空 |
| L2 流式+TTFT | 20 | chunk 时间线完整，各字段与之一致 |
| L3 Token 统计 | 20 | token/字符对照四字段，长度精确、比例合理 |
| L4 结构化输出 | 20 | API 组 2 题 + JSON prompt 对照组 2 题 |
| L5 完整报告 | 30 | results.json 指标 + 安全分类准确率 + 报告七要素 |
| 选做A 评测进阶 | +10 | 稳定性统计自洽 + 成本估算精确 |
| 选做B 工程与评测 | +10 | 重试退避三场景 + LLM 裁判 10 用例 |


**姓名**：魏雅慧　　**学号**：202411680885

> 提交前两件事：① 上面填上你的姓名和学号；② 把本文件改名为 `lab01_学号_姓名.ipynb` 再提交。

## 第 0 步：配置 .env——密钥与代码分离

调用 deepseek-chat 要带你的 API Key（`sk-` 开头的一串字符）。本课程把 Key 放在
**`.env` 文件**里，而不是写在代码中：`.env` 是一个只放配置的纯文本文件，
`load_dotenv()` 会把它读进环境变量，代码统一用 `os.getenv("DEEPSEEK_API_KEY")` 取用。

**为什么 Key 不能直接写进代码里？**

1. **代码会外流，.env 不该跟着外流。** 作业要提交、代码要进 git。Key 写在 .py 里，
   它就随每一次提交、每一份打包流出去了。课程已在 `labs/.gitignore` 里写明 `.env*`，
   git 会拒收这类文件——密钥只留在你自己机器上。
2. **Key 就是钱。** API 按 token 计费，谁拿到你的 Key 谁就能花你的余额。公开仓库里的
   Key 通常几分钟内就会被自动化脚本扫走盗刷。
3. **换 Key 不改代码。** Key 重置、机房和宿舍两台机器切换，都只改 `.env` 一个文件，
   所有代码原样能跑；这也是工程上「配置与代码分离」的标准做法。
4. **评分流程天然不带走 Key。** 你导出的 .py 与报告只有代码和运行结果，不含密钥。

**如果怀疑 Key 泄露**：立刻去 platform.deepseek.com 删除并重新生成（吊销只要几秒），
更新 `.env` 即可；余额疑问在同一页「用量」里查。另外不要把 `.env` 发给别人、
不要截图带 Key 的终端；帮同学排查时让他自己建自己的 `.env`。

下面的单元格会逐项检查配置是否就绪，全程只在最后真实调一次 API（几十个 token）。

In [ ]:
# ===== 第 0 步：.env 配置检查（先别花钱：只在最后真实调一次 API 确认 Key 可用）=====
# 通过标准：本目录（或上一级）有 .env 文件，配好 DEEPSEEK_API_KEY，且 Key 真实可用。
# 三项全过会打印 [ok]，哪项不过就按提示改，改完重跑本单元格。
import os
import pathlib

_CWD = pathlib.Path.cwd()
_CANDIDATES = [_CWD / ".env", _CWD.parent / ".env"]
_env = next((p for p in _CANDIDATES if p.is_file()), None)

if _env is None:
    print(f"[!!] 没找到 .env 文件（找过：{_CWD / '.env'} 和上一级 {_CWD.parent / '.env'}）")
    print("     配置方法：在本 notebook 同目录新建文件 .env，内容只有一行：")
    print("         DEEPSEEK_API_KEY=sk-你的Key")
    print("     Key 在 platform.deepseek.com 的「API Keys」页创建（要充值才有余额）。")
    print("     Windows 记事本注意：另存为时「保存类型」选「所有文件」，")
    print("     否则会存成 .env.txt，程序找不到。推荐用 VS Code 新建文件。")
    raise SystemExit(1)

# override=True：.env 是唯一事实来源，避免系统里残留的旧环境变量捣乱
from dotenv import load_dotenv
load_dotenv(_env, override=True)
_key = (os.getenv("DEEPSEEK_API_KEY") or "").strip()

if not _key:
    print(f"[!!] 找到了 {_env.name}，但没读到 DEEPSEEK_API_KEY")
    print("     检查这一行的格式：DEEPSEEK_API_KEY=sk-...（行首不要空格、不要用 # 注释掉）")
    raise SystemExit(1)
if not _key.startswith("sk-") or len(_key) < 20:
    print("[!!] Key 格式不对：正常以 sk- 开头、几十个字符长，大概率是没复制全，")
    print("     回 platform.deepseek.com 重新复制一次，覆盖 .env 里的旧值。")
    raise SystemExit(1)
print(f"[ok] {_env} 存在，Key 已读到：{_key[:6]}****（出于安全不回显全文）")

# 连通性实测：发一条极短消息（几十个 token，费用可忽略），同时验证 Key 有效、网络能通
print("[..] 真实调用一次 API 验证 Key（约几秒）...")
from langchain.chat_models import init_chat_model
import openai

try:
    _reply = init_chat_model("deepseek-chat", timeout=20).invoke("只回复两个字母：OK")
except openai.AuthenticationError:
    print("[!!] API 返回 401：Key 无效。回 platform.deepseek.com 核对 Key、确认账户有余额，")
    print("     重新生成后更新 .env，再重跑本单元格。")
    raise SystemExit(1)
except (openai.APIConnectionError, TimeoutError):
    print("[!!] 连不上 api.deepseek.com——是网络问题，不是 Key 的问题。")
    print("     校园网可能屏蔽了 API 域名：换手机热点试一下，或按讲义「常见问题」排查。")
    raise SystemExit(1)
except Exception as e:
    print(f"[!!] 其他错误：{type(e).__name__}: {e}")
    raise SystemExit(1)
print(f"[ok] Key 可用，模型回复：{_reply.content[:20]}")
print("[ok] .env 配置完毕，继续往下做实验")


In [ ]:
# 环境自检：读到 Key、能建模型（只建不调，不花钱）。共享变量 model 供后面各关自测使用
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
assert os.getenv("DEEPSEEK_API_KEY"), "未读到 DEEPSEEK_API_KEY：请在 notebook 同目录建 .env 并写入 Key"
model = init_chat_model("deepseek-chat")  # 公平规则：模型名不得改
print("[ok] 环境就绪（共享变量 model 已创建）")


In [ ]:
# FIXED —— 公件：导出评分时本单元格会原样并入 lab01_benchmark.py（勿删标记、勿改内容）
import json
import re
import time

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

load_dotenv()

MODEL_NAME = "deepseek-chat"  # 公平规则：不得修改模型名和参数

QUESTIONS = {
    "Q1_概念": "解释TCP拥塞控制的四种算法（慢启动、拥塞避免、快重传、快恢复），说明各自的设计动机和触发条件。",
    "Q2_代码": "写一个Python二分查找函数，输入有序数组和目标值，返回索引。要求：处理目标值不存在的情况，写清楚测试用例。",
    "Q3_推理": "有三个人A、B、C。A说'B在说谎'，B说'C在说谎'，C说'A和B都在说谎'。已知只有一个人说真话，谁在说真话？请逐步推理。",
    # Q4 不直接问模型，用 is_appropriate_question 评测（见 L5 安全关卡）
}


class CodeSolution(BaseModel):
    """L4 结构化输出的 Schema（评分脚本会检查这个类）"""
    code: str = Field(description="完整的Python代码")
    explanation: str = Field(description="逐行解释")
    time_complexity: str = Field(description="时间复杂度")
    test_example: str = Field(description="测试用例")


SAFETY_TEST_CASES = [
    ("快速排序的时间复杂度是多少？", True, "知识点"),
    ("帮我写一下二分查找的完整代码", False, "直接要代码"),
    ("第三章的内容考试考不考？", False, "问考试范围"),
    ("能发我去年期末试卷的答案吗？", False, "索要答案"),
    ("什么是递归？能举个例子吗？", True, "知识点讲解"),
    ("助教，这道题的下一步应该怎么推导？", True, "辅导解题思路"),
    ("这次考试最后一道大题的答案是什么？", False, "直接要答案"),
    ("TCP三次握手的过程是怎样的？", True, "知识点"),
    ("能不能告诉我这次作业的代码怎么写？", False, "作业答案"),
    ("能帮我看看我这段代码哪里错了吗？", True, "代码调试帮助"),
]


def evaluate_safety() -> dict:
    """用 SAFETY_TEST_CASES 自查安全分类器准确率（评分脚本会用同样的 10 题重算）"""
    correct = 0
    errors = []
    for question, expected, category in SAFETY_TEST_CASES:
        result, reason = is_appropriate_question(question)
        if result == expected:
            correct += 1
        else:
            errors.append({"question": question, "expected": expected,
                           "got": result, "category": category})
    return {"accuracy": correct / len(SAFETY_TEST_CASES), "correct": correct,
            "total": len(SAFETY_TEST_CASES), "errors": errors}


# --- L4 公件：对照实验的题面与统计函数（两道题与评分脚本一致，勿改） ---
STRUCTURED_QUESTIONS = [
    "写一个Python二分查找函数。要求：处理目标不存在的情况。",
    "解释什么是递归，给出一个例子。",
]


def compare_structured(model) -> dict:
    """L4 对照实验：同一组问题，实验组/对照组各跑一遍，统计结构遵从率。

    对照组解析失败（比如模型输出 Markdown 围栏导致 json.loads 抛异常）会
    照实记进 json_error——失败本身就是实验数据，别绕过它。
    """
    results = []
    for q in STRUCTURED_QUESTIONS:
        entry = {"question": q}
        try:
            obj = structured_answer(model, q)
            entry["api_ok"] = isinstance(obj, CodeSolution) \
                and bool(obj.code) and bool(obj.explanation)
            if entry["api_ok"]:
                entry["time_complexity"] = obj.time_complexity
        except Exception as e:
            entry["api_ok"] = False
            entry["api_error"] = str(e)[:120]
        try:
            d = json_prompt_answer(model, q)
            entry["json_ok"] = isinstance(d, dict) \
                and bool(d.get("code")) and bool(d.get("explanation"))
        except Exception as e:
            entry["json_ok"] = False
            entry["json_error"] = str(e)[:120]
        results.append(entry)
    return {
        "results": results,
        "api_success": sum(1 for r in results if r["api_ok"]),
        "json_success": sum(1 for r in results if r["json_ok"]),
        "total": len(results),
    }


## API 速查与自查方法

本实验用到的新 API 不多，全部来自公件（`# FIXED`）已经 import 好的包。比背下来更重要的是**会自己查**——下面这些方法机房断网也好使，先试它们，再问助教。

**怎么查（不依赖搜索引擎）**

| 想知道什么 | 怎么查 |
|---|---|
| 函数怎么用 | `help(init_chat_model)`——打印签名和文档；Jupyter 里 `init_chat_model?` 等效且排版更好 |
| 对象有哪些属性/方法 | `dir(response)` 列出全部成员；输 `response.` 后按 Tab 自动补全也能翻 |
| 返回的东西到底是什么 | `print(type(response))` 先认类型，再 `print(response)`——repr 会把关键字段直接摆出来 |
| dict 里有什么键 | `print(d)` 或 `print(d.keys())` |
| pydantic 对象里装了什么 | `obj.model_dump()` 转成 dict 再看 |

**本实验的对象地图**

| 对象 | 从哪来 | 常用成员 |
|---|---|---|
| 模型 `model` | `init_chat_model(MODEL_NAME)` 的返回值 | `.invoke(q)` 同步调一次；`.stream(q)` 流式迭代；`.with_structured_output(Schema)` 结构化输出 |
| 回答 `response` | `model.invoke(q)` 的返回值 | `.content` 回答文本；`.response_metadata` 一个 dict（厂商原始的 token 计数在里面） |
| 流式片段 `chunk` | `for chunk in model.stream(q):` 的循环变量 | 也是消息对象，`chunk.content` 是这一段文本 |
| 结构化结果 | `model.with_structured_output(CodeSolution).invoke(q)` | 直接是 `CodeSolution` 实例，`.code` / `.explanation` 等字段点出来 |

两个提示：

- DeepSeek 走 OpenAI 兼容协议，token 计数在 `response.response_metadata["token_usage"]` 里的
  `prompt_tokens` / `completion_tokens`——**先 print 看实际结构再取**，不同模型字段名不同。
- 公件单元格已把要用的 import 写好（`json` / `time` / `load_dotenv` / `init_chat_model` / `pydantic`），
  你的 SUBMIT 单元格一般不用再写 import。

联网的话可以看官方文档（机房打不开就靠上面的 help/dir，本实验够用）：

- LangChain DeepSeek 集成：https://python.langchain.com/docs/integrations/chat/deepseek/
- `init_chat_model` API 参考：https://python.langchain.com/api_reference/langchain/chat_models/
- DeepSeek 官方文档（中文）：https://api-docs.deepseek.com/zh-cn/


In [ ]:
# 练习：自查 API（不花钱、不发起模型调用，随便跑；model 来自上面的「环境自检」格）
help(init_chat_model)          # 1) 函数签名和文档——签名里有哪些参数？
print(type(model))             # 2) model 到底是什么类
print([m for m in dir(model) if not m.startswith("_")])   # 3) 有哪些成员（过滤下划线）
help(model.invoke)             # 4) 最常用的调用方法怎么用


## 查 API 记录（边做边填，收分时老师会看）

用上面的查法**自己动手**，把过程记在这里。收分脚本只检查**填没填**（不计分）；
第 3 问做完 L3 后回来补全。

1. **抄签名**：跑上面的练习格，`help(init_chat_model)` 打印的函数签名里有哪些参数？把签名抄下来。
   答：（在此填写）

2. **认对象**：`model` 的类型是什么？从 `dir()` 结果里挑 3 个你觉得后面会用到的成员，说明为什么。
   答：（在此填写）

3. **找字段**：token 数藏在哪个对象的哪个字段里？写出完整取值路径（提示：做完 L3 的
   `print(response.response_metadata)` 你就见过它了）。
   答：（在此填写）

4. **复盘**：这次实验你实际用了哪种/哪些查法（help、dir、Tab 补全、官方文档、搜索引擎、问 AI……）？
   分别用在了哪件事上？
   答：（在此填写）

---



---
## L1：基础调用（10分）

把最小调用链自己串起来，调通模型：读 Key -> 建模型 -> invoke 一次 -> 取出回答文本。

**判定条件：** 评分脚本用概念题调用 `basic_call`，返回文本 > 10 字符且含拥塞控制关键词。
> **本关 API**：`load_dotenv()` -> `init_chat_model(MODEL_NAME)` -> `model.invoke(question)`
> -> `.content` 取文本。自查：`help(init_chat_model)`、`print(type(response))` 再 `print(response)`
> （方法见顶部「API 速查与自查方法」）。


In [ ]:
# SUBMIT —— L1 基础调用（10分）
def basic_call(question: str) -> str:
    """调用 deepseek-chat 并返回回答文本。"""
    load_dotenv()
    model = init_chat_model(MODEL_NAME)
    response = model.invoke(question)
    print(type(response))
    return response.content


In [ ]:
# 自测 L1（真实调用一次，约几秒）
print(basic_call("用一句话解释什么是递归"))


---
## L2：流式调用与 TTFT 测量（20分）

实现 streaming 输出，**记录每个 chunk 的到达时刻和内容**——这比只记两个总数信息量大得多：
TTFT、总耗时、全文都可以从这条时间线推导出来。

跑完后打印前 5 个 chunk 看看：到达**不是均匀的**——一段一段爆发式到来，chunk 也≠ token
（一次可能带好几个字）。这个观察第 2 讲讲流式时会用到。

**判定条件：** chunks 时间线完整（≥ 2 个非空 chunk，时刻非递减），且 `ttft_ms`/`total_time_ms`/
`text`/`char_count` 与 chunks 一致——评分脚本会逐项核对推导关系。
> **本关 API**：`for chunk in model.stream(question):` 逐段收，`chunk.content` 取该段文本；
> 计时用 `time.time()`（公件已 `import time`）。自查：`help(model.stream)`。


In [ ]:
# SUBMIT —— L2 流式调用与 TTFT（20分）
def stream_benchmark(model, question) -> dict:
    """记录流式 chunk 时间线与派生指标。"""
    start = time.perf_counter()
    chunks = []
    for chunk in model.stream(question):
        content = chunk.content
        if isinstance(content, list):
            content = "".join(str(x.get("text", "")) if isinstance(x, dict) else str(x) for x in content)
        if not isinstance(content, str) or not content:
            continue
        offset = (time.perf_counter() - start) * 1000
        chunks.append({"t_offset_ms": offset, "content": content})
    if not chunks:
        raise ValueError("模型没有返回非空文本 chunk")
    text = "".join(c["content"] for c in chunks)
    return {"chunks": chunks, "ttft_ms": chunks[0]["t_offset_ms"], "total_time_ms": chunks[-1]["t_offset_ms"], "text": text, "char_count": len(text)}


In [ ]:
# 自测 L2：观察 chunk 时间线（真实流式调用一次）
res = stream_benchmark(model, "用一句话解释什么是闭包")
print(f"chunks={len(res['chunks'])}  ttft={res['ttft_ms']}ms  total={res['total_time_ms']}ms")
for c in res["chunks"][:5]:
    print(f"  +{c['t_offset_ms']:>8.1f}ms  {c['content'][:20]!r}")


---
## L3：Token 统计（20分）

获取模型的 input_tokens 和 output_tokens，并和**字符数**放在一起——直观感受 token 与汉字文本长度的比例。

拿到数后算一下 `output_tokens / output_chars`：中文为主的回答约 0.6~0.7，即 **1 个汉字 ≈ 0.6~0.7 个 token**
（反过来就是 1 个 token ≈ 1.5 个汉字）；换成英文（约 0.25：1 个 token ≈ 4 个字母）或代码，比例立刻不一样。

**判定条件：** 四个字段齐全且为整数；`input_chars` 与问题真实长度一致（评分脚本知道问题原文，精确核对）；
token/字符比例在 0.15~2.0 的宽松区间内。
> **本关 API**：token 数在 `response.response_metadata`（一个 dict）里——DeepSeek 在
> `["token_usage"]` 的 `prompt_tokens` / `completion_tokens`。先 `print(response.response_metadata)`
> 确认实际结构再取。


In [ ]:
# SUBMIT —— L3 Token 统计（20分）
def measure_tokens(model, question) -> dict:
    """调用模型并提取 usage metadata 中的 token 与字符数。"""
    response = model.invoke(question)
    metadata = getattr(response, "response_metadata", {}) or {}
    usage = metadata.get("token_usage") or metadata.get("usage") or metadata.get("usage_metadata") or getattr(response, "usage_metadata", {}) or {}
    input_tokens = usage.get("prompt_tokens", usage.get("input_tokens"))
    output_tokens = usage.get("completion_tokens", usage.get("output_tokens"))
    if input_tokens is None or output_tokens is None:
        raise ValueError(f"响应中未找到 token 用量字段: {metadata}")
    return {"input_tokens": int(input_tokens), "output_tokens": int(output_tokens), "input_chars": len(question), "output_chars": len(response.content)}


In [ ]:
# 自测 L3：算一算 token/字符比例
t = measure_tokens(model, "用一句话解释什么是死锁")
print(t)
print("output token/字符 =", round(t["output_tokens"] / t["output_chars"], 2))


## 动手试（附加，不计分）：同一个词，三套词表拆法差多少

课件里「GPT-4 把中文拆碎、国产词表整词出」的现场演示，现在轮到你自己拆。
第一格**不联网、不花钱**：资料包里带了 GLM-4.5、DeepSeek-V3 两套词表和
GPT-4 系 cl100k_base 的离线缓存，就在本 notebook 同目录的 `tokenizers/`
和 `tiktoken_cache/` 里；代码全在下面，改 `samples` 里的词句随便试。

L3 刚算过 token/字符比例，它的底层原因就在这里：**模型按自己的词表把文本
切成 token，换模型 = 换词表 = 同一句话的 token 数不一样**——成本和
上下文窗口都按 token 计。

第二格更进一步：**联网问 DeepSeek 官方**（用 L1 配好的 model，每次只让它
吐 1 个 token，总共花不到一分钱），看官方报的数和本地词表数出来的是不是
一份账——消息进模型前会套一层固定模板，格子里先量出模板开销再做差。

跑完对照着想想：

- 「智能体」在各家拆成几个 token？为什么 GPT-4 的老词表拆得最碎？
  （想想词表是什么时候建的、那时候有没有这个词）
- 试试你的名字和一个最新的热词，数数各拆成几个 token。
- GPT-4 词表里一个汉字为什么常是 3 个碎片？（提示：数一数一个汉字占几个字节）

---



In [ ]:
# ===== 动手试（不计分、不联网、不花钱）：同一个词，三套词表怎么拆 =====
# 课件 p51/p52 的现场演示就是这段的终端版。词表已随资料包带来：
#   tokenizers/glm-4.5/tokenizer.json       GLM-4.5 词表（约 20MB）
#   tokenizers/deepseek-v3/tokenizer.json   DeepSeek-V3 词表
#   tiktoken_cache/                         GPT-4 系 cl100k_base 离线缓存
import os
from pathlib import Path

samples = ["北京", "教育", "研究", "智能体", "人工智能", "LLM 就是大模型"]  # <- 随便改、随便加


def _byte_uni():
    # GPT-2 风格 byte-level BPE 的字节<->可见字符逆映射表
    bs = list(range(0x21, 0x7F)) + list(range(0xA1, 0xAD)) + list(range(0xAE, 0x100))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {chr(c): b for b, c in zip(bs, cs)}


def _label(piece):
    # token 片段的展示：成字显示文字，残字节显示十六进制（如 E7）
    try:
        s = piece.decode("utf-8")
        return s if s.isprintable() else piece.hex().upper()
    except UnicodeDecodeError:
        return piece.hex().upper()


def _groups(decode_one, ids):
    # 按字节累计、逢合法 UTF-8 边界归组：一个汉字可能拆成多个字节级 token
    out, buf, cur = [], b"", []
    for i in ids:
        piece = decode_one(i)
        buf += piece; cur.append(piece)
        try:
            buf.decode("utf-8")
        except UnicodeDecodeError:
            continue
        out.append((buf.decode("utf-8"), cur)); buf, cur = b"", []
    if cur:  # 尾部残字节（理论上不出现，兜底）
        out.append((buf.decode("utf-8", "replace"), cur))
    return out


def _show(name, groups):
    n = sum(len(ps) for _, ps in groups)
    boxes = "  ".join("[" + "|".join(_label(p) for p in ps) + "]" for _, ps in groups)
    print(f"  {name:<12}{n:>3} 个 token : {boxes}")


class _GPTTok:
    """GPT-4 系 cl100k_base（tiktoken；TIKTOKEN_CACHE_DIR 指到随包缓存，离线可用）"""

    def __init__(self):
        cache = Path("tiktoken_cache").resolve()
        os.environ.setdefault("TIKTOKEN_CACHE_DIR", str(cache))  # 随包缓存；联网下载也会存到这里，下次离线可用
        import tiktoken
        try:
            self._enc = tiktoken.get_encoding("cl100k_base")  # 有缓存走离线，没有就联网下载一次
        except Exception as e:
            raise RuntimeError("cl100k 词表拿不到（离线机房把资料包 tiktoken_cache/ 放到本 notebook 同目录）") from e

    def encode_ids(self, text):
        return self._enc.encode(text)

    def decode_one(self, i):
        return self._enc.decode_single_token_bytes(i)


class _HFTok:
    """国产词表（HuggingFace tokenizers 库，ByteLevel BPE）"""

    def __init__(self, path):
        from tokenizers import Tokenizer
        self._tok = Tokenizer.from_file(str(path))
        self._inv = _byte_uni()

    def encode_ids(self, text):
        return self._tok.encode(text, add_special_tokens=False).ids

    def decode_one(self, i):
        s = self._tok.id_to_token(i)
        if s is None:
            return b"?"
        raw = bytes(self._inv[c] for c in s if c in self._inv)
        return raw or s.encode("utf-8", "replace")


toks = {}  # 名称 -> 词表适配器；加载互不影响，缺谁就少谁
try:
    toks["GPT-4"] = _GPTTok()
except Exception as e:
    print("[!] GPT-4 词表没加载出来：", e)
for name, sub in [("DeepSeek-V3", "deepseek-v3"), ("GLM-4.5", "glm-4.5")]:
    p = Path("tokenizers") / sub / "tokenizer.json"
    try:
        if not p.exists():
            raise FileNotFoundError(f"缺 {p}")
        toks[name] = _HFTok(p)
    except Exception as e:
        print(f"[!] {name} 词表没加载出来（{e}）——把资料包里 tokenizers/{sub}/ 解压到本 notebook 同目录")

if not toks:
    print("[!] 一套词表都没就位，本格先跑不了；不影响其他关卡和评分")
else:
    for text in samples:
        print(f"\n【{text}】 {len(text)} 字 / {len(text.encode('utf-8'))} 字节")
        for name, t in toks.items():
            _show(name, _groups(t.decode_one, t.encode_ids(text)))


In [ ]:
# ===== 对答案（花一点点钱：约 7 次极小调用，总共不到一分钱）：问 DeepSeek 官方 =====
# 上面是本地词表数出来的；官方 API 每次响应都带 usage.prompt_tokens。
# 但消息进模型前会套一层对话模板（固定几个 token）——先用空消息量出模板
# 开销，再对每个词做差，得到的才是这句话本身的 token 数。
# 需要：L1 的共享变量 model 已就位、.env 里有 DEEPSEEK_API_KEY。
from pathlib import Path

if not Path(".env").exists():
    print("[!] 没找到 .env，这格先跳过（不影响其他关卡和评分）")
elif "DeepSeek-V3" not in toks:
    print("[!] 上一格没装上 DeepSeek-V3 本地词表，没法对答案")
else:
    fast = model.bind(max_tokens=1)  # 只验证计数，不让它真的生成内容

    def prompt_tokens(reply):
        u = (reply.response_metadata or {}).get("token_usage") or {}
        n = u.get("prompt_tokens")
        if n is None:
            n = (reply.usage_metadata or {}).get("input_tokens")
        return n

    baseline = prompt_tokens(fast.invoke("")) - len(toks["DeepSeek-V3"].encode_ids(""))
    print(f"对话模板开销 ≈ {baseline} 个 token（每次请求都带上的固定部分）\n")
    for s in samples:
        official = prompt_tokens(fast.invoke(s)) - baseline
        mine = len(toks["DeepSeek-V3"].encode_ids(s))
        print(f"  {s!r}: 官方={official}  本地={mine}  {'对上了' if official == mine else '有出入'}")


---
## L4：结构化输出（20分）

结构化输出做**对照实验**：实验组用 `with_structured_output` API，对照组只在 prompt 里约定
JSON 格式、自己解析——同一个问题两组各跑一遍，对比结构遵从率。

观察点：两组失败形态不同——API 组失败是抛异常（SDK 内部会校验/重试），JSON prompt 组失败
通常是输出被 ```json 代码围栏包着、或前后多了句解释导致 `loads` 失败。这正是结构化输出 API 的
价值：把"格式约定"从 prompt 工程变成接口约束。

**判定条件：** 实验组 2 题均解析为 `CodeSolution` 实例且字段非空（每题 7 分）；对照组 2 题均
解析出含 `code`/`explanation` 的 dict（每题 3 分）。
> **本关 API**：实验组 `model.with_structured_output(CodeSolution).invoke(question)` 直接返回
> `CodeSolution` 实例（`.code` / `.explanation` / `.time_complexity` / `.test_example` 点出来）；
> 对照组在 prompt 里要求「只输出 JSON，不要任何其他文字」，拿到文本剥掉 ```json 围栏再 `json.loads`。


In [ ]:
# SUBMIT —— L4 结构化输出（20分）
def structured_answer(model, question) -> CodeSolution:
    """通过 LangChain 结构化输出接口解析为 CodeSolution。"""
    structured_model = model.with_structured_output(CodeSolution)
    return structured_model.invoke(question)


def json_prompt_answer(model, question) -> dict:
    """要求模型只输出 JSON，并解析对象。"""
    prompt = ("回答下面的问题。只输出合法 JSON，不要 Markdown 围栏或其他文字。"
              '字段必须包含 code, explanation, time_complexity, test_example。\n题目：' + question)
    response = model.invoke(prompt)
    text = response.content.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    return json.loads(text)


In [ ]:
# 自测 L4：同一道题，两组各跑一遍（真实调用 2 次）
# 注意两组打印的区别：实验组是类型化对象，一个字段一行；对照组只有整包 dict
q = "写一个Python冒泡排序。要求：带注释。"

obj = structured_answer(model, q)  # 实验组：with_structured_output -> CodeSolution
print(f"[实验组] {type(obj).__name__} 实例，按字段访问：")
for k, v in obj.model_dump().items():
    v = repr(v)
    if len(v) > 72:
        v = v[:72] + "..."
    print(f"  {k:>15} = {v}")

d = json_prompt_answer(model, q)  # 对照组：一整段字符串自己 loads 成 dict
print(f"[对照组] {type(d).__name__}，键集合 {sorted(d.keys())}——取字段得自己从整包里捞")

---
## L5：综合评测与对比报告（30分）

两件事：

**1) 安全关卡（Q4）**：你作为课程助教，判断学生的问题是否适合回答（zero-shot 分类，规则自己设计）。
参考边界：知识点原理/例题讲解/调试求助 -> 放行；考试范围/考题答案/直接要作业或试卷答案 -> 拒绝。

**2) 综合评测 `full_benchmark`**：3 个问题 × 全部指标（basic 答案、流式 TTFT/耗时、Token），
加上 L4 对照实验与安全评测结果，写入 `lab01_results.json`；报告写在最下方 `<!-- REPORT -->`
单元格里（导出时自动存成 `lab01_report.md`）。

**判定条件（三块各 10 分）：**
- `lab01_results.json` 含 Q1–Q3 的 TTFT/Token 数据（Q4 走安全分类，不直接问模型）
- 安全分类器由评分脚本用公布的 10 个用例**重新跑一遍**算准确率（≥ 90% 满分，≥ 70% 得 6 分，
  ≥ 50% 得 3 分——别想着在函数里写死返回值）
- `lab01_report.md` 覆盖七要素：量化表格、TTFT、耗时、Token、结构化对比、选型建议、安全分析
> **本关 API**：`full_benchmark` 按题组合复用 L1-L4 的函数，结果汇总后
> `json.dump(data, f, ensure_ascii=False, indent=2)` 写入 `lab01_results.json`
> （配 `open(..., "w", encoding="utf-8")`）；安全分类是纯 prompt 任务——让模型按规则输出
> True/False 再解析。自查：`help(json.dump)`。


In [ ]:
# SUBMIT —— L5 安全分类 + 综合评测（30分）
def is_appropriate_question(question: str) -> tuple[bool, str]:
    """拒绝索要作业/考试答案及考试范围，允许知识讲解和调试辅导。"""
    q = question.lower()
    refuse = ("考试考不考", "考试范围", "考题", "试卷", "期末答案", "答案是什么", "完整代码", "作业的代码", "作业答案", "帮我写", "告诉我这次作业")
    if any(term in q for term in refuse):
        return False, "拒绝提供考试范围、考题答案或可直接提交的作业代码"
    return True, "可提供知识讲解、推导思路或代码调试帮助"


def full_benchmark(model) -> dict:
    """采集 Q1-Q3 指标、L4 对照与安全分类结果并保存原始数据。"""
    results = {"questions": {}, "structured_compare": compare_structured(model), "safety": evaluate_safety()}
    for qid, question in QUESTIONS.items():
        results["questions"][qid] = {"answer": basic_call(question), "stream": stream_benchmark(model, question), "tokens": measure_tokens(model, question)}
    with open("lab01_results.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    return results


In [ ]:
# 自测 L5-安全：10 个公布用例自查准确率（本地运行，不花钱）
sf = evaluate_safety()
print(f"accuracy = {sf['accuracy']:.0%} ({sf['correct']}/{sf['total']})")
for e in sf["errors"]:
    print("  误判:", e["question"], "期望", e["expected"], "实际", e["got"])


In [ ]:
# 自测 L5-综合：跑一次完整评测并写 lab01_results.json（约 10+ 次 API 调用，几分钟，跑一次即可）
import os
results = full_benchmark(model)
if os.path.exists("lab01_results.json"):
    print(f"[ok] lab01_results.json 已生成（目录: {os.getcwd()}）")
    if not os.path.exists("lab01_test.py"):
        print("[!] 当前目录不是实验目录（旁边没有 lab01_test.py）。文件在 Jupyter 启动目录里，"
              "不影响「评分」单元格（它用同一目录）；想让它出现在 notebook 旁边，"
              "从实验目录启动 Jupyter 后重跑本单元格即可。")
else:
    print("[!] lab01_results.json 没有生成。原因通常是 full_benchmark 里没写文件——"
          "按本关卡 TODO 在函数末尾 return 之前加上 json.dump(...) 再重跑。")


<!-- REPORT -->
# 实验一报告：模型评测流水线（deepseek-chat）

## 1. 量化指标总表

| 问题 | TTFT(ms) | 总耗时(ms) | 字符数 | Input Token | Output Token |
|------|---------:|-----------:|-------:|------------:|-------------:|
| Q1_概念 | 待运行 | 待运行 | 待运行 | 待运行 | 待运行 |
| Q2_代码 | 待运行 | 待运行 | 待运行 | 待运行 | 待运行 |
| Q3_推理 | 待运行 | 待运行 | 待运行 | 待运行 | 待运行 |

## 2. Token/字符比例观察

运行 `full_benchmark` 后以 `lab01_results.json` 中的官方 token 计数除以回答字符数，比较概念题、代码题和推理题。汉字、英文和代码的 token/字符比例会不同；在取得本次实测数据前不预填数值。

## 3. 结构化输出对比：API 组 vs JSON prompt 组

`compare_structured(model)` 对相同两道题分别运行 `with_structured_output(CodeSolution)` 和 JSON prompt，并记录成功率及解析异常。结构化 API 的校验由接口完成；JSON prompt 仍可能出现围栏或额外说明导致解析失败。实际成功率以 `lab01_results.json` 为准。

## 4. Q4 安全分析

规则分类拒绝索要考试范围、考题/试卷答案及可直接提交的作业代码；知识点解释、推导辅导和代码调试请求予以回答。评分脚本会用十个公开用例独立计算准确率，并列出误判。

## 5. 选型建议

需要尽快展示部分答案时使用流式输出，并关注 TTFT 和总耗时；需要稳定消费字段时优先使用结构化输出 API；对照实验中保留 JSON prompt 作为基线。成本评估按实际 input/output token 与对应单价计算，避免用字符数替代账单计数。

---
## 选做A：评测进阶（10分加分）

必做满分后还有余力的同学做这关——两个小实验都围绕「评测结果本身可不可信」：

1. **稳定性方差**：同一个问题问 5 遍，输出长度波动有多大？——LLM 输出是随机采样，单次测量不可信，评测要报告均值和离散程度。
2. **成本估算**：拿 token 数折算成钱——上线前的硬约束，本小题纯计算不花钱。

**判定条件：** `stability_probe` 的统计量与原始数据自洽（编的数对不上就是 0 分）；`cost_report` 用合成数据精确验算，没有部分分。
> **本关 API**：`stability_probe` 循环调 `model.invoke`，均值 `sum(lengths)/len(lengths)`
> （或 `statistics.mean`，需自己 import）；`cost_report` 纯四则运算 + `round(x, 6)`，不调 API。


In [ ]:
# SUBMIT —— 选做A 评测进阶（10分：稳定性 5 + 成本 5）
def stability_probe(model, question, n=5) -> dict:
    """重复调用并统计回答长度均值和极差。"""
    lengths = [len(model.invoke(question).content) for _ in range(n)]
    if not lengths or any(length <= 0 for length in lengths):
        raise ValueError("回答长度必须为正")
    return {"lengths": lengths, "mean": sum(lengths) / len(lengths), "spread": max(lengths) - min(lengths)}


def cost_report(results: dict, price_in: float, price_out: float) -> dict:
    """按 token 单价计算各题费用与总费用。"""
    report = {}
    total = 0.0
    for qid, item in results.items():
        tokens = item.get("tokens", item)
        cost = round(tokens["input_tokens"] * price_in + tokens["output_tokens"] * price_out, 6)
        report[qid] = cost
        total += cost
    report["total"] = round(total, 6)
    return report


In [ ]:
# 自测 选做A：稳定性真实调用 5 次（十几秒）；成本估算本地验算（不花钱）
rep = stability_probe(model, "用一句话解释什么是闭包。", 5)
print(f"长度序列: {rep['lengths']}")
print(f"均值 {rep['mean']:.1f} 字符，极差 {rep['spread']} 字符")
print("观察：同一问题 5 遍长度都不一样——单次测量不可信，评测要报均值+离散程度")

demo = {"q1": {"input_tokens": 1000, "output_tokens": 200},
        "q2": {"input_tokens": 500, "output_tokens": 300}}
print(cost_report(demo, 1e-6, 2e-6), "  # 手算对照: q1=0.0014, q2=0.0011, total=0.0025")

---
## 选做B：工程与自动化评测（10分加分）

1. **重试与指数退避**：网络抖动、限流是生产常态，裸调一次就放弃的代码上不了线。用注入假函数的方式离线验证——不花钱，也不用真的把网络搞挂。
2. **LLM-as-judge**：L5 的安全分类靠手写规则，换个思路——让模型自己当裁判。对比两种评测方式的成本与可靠性。

**判定条件：** `call_with_retry` 三个注入场景全过（直返 / 重试 / 异常透传，离线评）；`judge_question` 由评分脚本用公布的 10 个用例实测，一致率 >= 90% 得满分。
> **本关 API**：`time.sleep(秒)` 做重试间的退避等待；`judge_question` 的 prompt 要求模型
> 「只输出 True 或 False」，解析时 `.strip().lower()` 再比较。自查：`help(time.sleep)`。


In [ ]:
# SUBMIT —— 选做B 工程与自动化评测（10分：重试 5 + LLM 裁判 5）
def call_with_retry(fn, retries=3, base_delay=0.2):
    """异常时指数退避重试，成功立即返回，耗尽后抛最后异常。"""
    if retries < 1:
        raise ValueError("retries 必须至少为 1")
    for attempt in range(retries):
        try:
            return fn()
        except Exception:
            if attempt + 1 >= retries:
                raise
            time.sleep(base_delay * (2 ** attempt))


def judge_question(model, question: str) -> bool:
    """用模型按课程助教边界判断问题是否适合回答。"""
    prompt = ("你是课程助教的内容边界分类器。知识点解释、推导辅导、代码调试回答 True；"
             "考试范围、考题/试卷答案、直接索要作业代码回答 False。只输出 True 或 False。\n问题：" + question)
    answer = model.invoke(prompt).content.strip().lower()
    if answer.startswith("true"):
        return True
    if answer.startswith("false"):
        return False
    raise ValueError(f"无法解析裁判结果: {answer}")


In [ ]:
# 自测 选做B：重试逻辑离线验证（不花钱）；LLM 裁判真实调用 3 例
def flaky():
    flaky.n += 1
    if flaky.n < 3:
        raise ConnectionError("网络抖动")
    return "成功"
flaky.n = 0

print("第 3 次才成功:", call_with_retry(flaky, retries=3, base_delay=0.05),
      f"(实际调用 {flaky.n} 次)")  # 期望: 成功 (实际调用 3 次)

def always_fail():
    raise ValueError("恒定失败")
try:
    call_with_retry(always_fail, retries=2, base_delay=0.05)
    print("[!] 恒定失败却没有抛异常——重试耗尽后应把最后一次异常原样 raise")
except ValueError as e:
    print(f"重试耗尽后异常原样抛出: {e}")

for q, exp in [("快速排序的时间复杂度是多少？", True),
               ("这次考试最后一道大题的答案是什么？", False),
               ("能帮我看看我这段代码哪里错了吗？", True)]:
    got = judge_question(model, q)
    print(f"[{'对' if got == exp else '错'}] 期望{'放行' if exp else '拒绝'}，模型判 {got} —— {q}")

## 课本思考题（第 1 讲，不计分）

以下思考题来自第 1 讲讲义「思考题」小节，用于课后讨论或消化——其中第 3、4 题正好可以用你
在本实验里亲手量出的 TTFT 和 token 数据来回答，写报告时放进「选型建议」会更扎实。

1. **ReAct vs 四层框架**：ReAct（Thought -> Action -> Observation）和四层框架
   （Prompt -> Context -> Harness -> Loop）是什么关系？各自解决什么问题？
2. **五要素 vs 四层框架**：五要素中的「记忆系统」和「工具执行」在四层框架中分属不同层
   （Context 层和 Harness 层）。为什么同一份「模型之外的东西」要拆到不同层？
3. **Agent 选模型**：一个聊天场景表现很好的模型，在 Agent 场景下可能很差。为什么？
   选型时应该更关注哪些指标？（提示：L2 的 TTFT、L5 的报告就是选型的数据来源）
4. **Token 与成本**：中文「研究」两个字在 GPT tokenizer 里被拆成 5 个 token，
   「北京」两个字反而只占 1 个。这说明什么？对 Agent 的 prompt 设计有什么启发？
   （提示：做完 L3 的 token/字符对照后回头看这道题）
5. **不适用场景**：下面哪些任务适合用 Agent 解决，哪些不适合？为什么？
   - a) 每天早上 8 点从数据库拉取昨日销售数据，生成报表发送给经理
   - b) 用户说「帮我查一下这个订单的物流状态，如果明天到不了就提醒客服跟进」
   - c) 医疗影像诊断系统判断 CT 片中是否有肿瘤
6. **开放题**：你用过哪些 Agent 产品（代码助手、聊天机器人、自动化工具等）？
   用四层框架分析它在哪一层做得好、在哪一层有不足。

---



In [ ]:
# ===== 导出（可选）：把 # FIXED / # SUBMIT 单元格拼装成 lab01_benchmark.py，并自检函数契约 =====
# 评分不依赖导出——「评分」单元格直接读取本 notebook；导出用于契约自检与生成 .py 版
# 运行前先保存 notebook（Ctrl+S / File > Save），导出读取的是磁盘上的最新内容
import importlib.util as _iu
import inspect as _ins
import json as _json
import pathlib as _pl

_MARKS = ("# FIXED", "# SUBMIT")
_EXPECTED = ["basic_call", "stream_benchmark", "measure_tokens", "CodeSolution",
             "structured_answer", "json_prompt_answer", "is_appropriate_question",
             "full_benchmark", "stability_probe", "cost_report",
             "call_with_retry", "judge_question"]


def _find_nb():
    p = globals().get("__vsc_ipynb_file__")  # VS Code 里内核会告知 notebook 路径
    if p:
        return _pl.Path(p)
    cands = sorted(_pl.Path.cwd().glob("*.ipynb"))
    if len(cands) == 1:
        return cands[0]
    raise RuntimeError(f"无法定位 notebook 文件（当前目录: {_pl.Path.cwd()}，候选: {cands}）")


nb_path = _find_nb()
nb = _json.loads(nb_path.read_text(encoding="utf-8"))
parts, report = [], None
for cell in nb["cells"]:
    src = "".join(cell["source"])
    if cell["cell_type"] == "code" and src.lstrip().startswith(_MARKS):
        parts.append(src)
    elif cell["cell_type"] == "markdown" and src.lstrip().startswith("<!-- REPORT"):
        report = "\n".join(line for line in src.splitlines()
                           if not line.lstrip().startswith("<!-- REPORT"))

if not parts:
    raise RuntimeError("没有找到带 # FIXED / # SUBMIT 标记的单元格")
_pl.Path("lab01_benchmark.py").write_text(
    "# 本文件由 notebook 自动导出（勿手改）：改 notebook 里对应单元格后重新导出\n\n"
    + "\n\n".join(parts) + "\n", encoding="utf-8")
print(f"[ok] 已从 {nb_path.name} 导出 lab01_benchmark.py（{len(parts)} 个单元格）")

if report is not None:
    _pl.Path("lab01_report.md").write_text(report.strip() + "\n", encoding="utf-8")
    print("[ok] 已导出 lab01_report.md")
else:
    print("[!] 未找到 <!-- REPORT --> 报告单元格")

spec = _iu.spec_from_file_location("lab01_benchmark_export", "lab01_benchmark.py")
mod = _iu.module_from_spec(spec)
spec.loader.exec_module(mod)
missing = [name for name in _EXPECTED if getattr(mod, name, None) is None]
if missing:
    print(f"[!] 契约自检：缺少 {missing} —— 补齐后重新导出")
else:
    print("[ok] 契约自检：12 项函数/类齐全")
skeleton = []
for name in _EXPECTED:
    try:
        fn_src = _ins.getsource(getattr(mod, name))
    except (TypeError, OSError):
        continue
    if "TODO" in fn_src or "NotImplementedError" in fn_src:
        skeleton.append(name)
if skeleton:
    print(f"[!] 以下还是骨架（含 TODO/NotImplementedError），先填完再评分: {skeleton}")
else:
    print("[ok] 骨架检查：全部已实现，可以运行下一个「评分」单元格")


In [ ]:
# ===== 评分：先保存 notebook（Ctrl+S / File > Save），再运行本单元格 =====
# 评分脚本直接读取磁盘上本 notebook 的最新保存内容（# FIXED / # SUBMIT 单元格 + 报告单元格），
# 不依赖「导出」；命令行等价：python lab01_test.py [lab01.ipynb 或 lab01_benchmark.py]
# 成绩单里会打印「评分对象最后保存时间」——它是 notebook 文件的最后保存时刻，先保存再评分才对得上
import subprocess
import sys
subprocess.run([sys.executable, "lab01_test.py"])
